# Pipeline de Transformação: devolucoes

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.services.spark_session import get_spark_session, close_spark_session
import src.modules.transform_utils as transform
import src.modules.utils as utils

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("TransformDevolucoes")

# Leitura dos dados da camada Bronze

In [ ]:
# Caminho da tabela Bronze no MinIO
bronze_path = "s3a://bronze/devolucoes"

# Lê os dados da Bronze
try:
    df_devolucoes = spark.read.parquet(bronze_path).withColumnRenamed("data_carga", "data_carga_bronze")
except Exception as e:
    print(f"Erro ao ler a camada Bronze: {e}")

print(f"Total de registros na camada Bronze: {df_devolucoes.count()}")
df_devolucoes.printSchema()
df_devolucoes.limit(5).toPandas()

# Aplica TRIM nas colunas de texto

In [ ]:
text_cols = ["motivo_devolucao", "status_devolucao"]
df_devolucoes = transform.trim_columns(df_devolucoes, text_cols)

df_devolucoes.select(*text_cols).distinct().limit(5).toPandas()

# Cria colunas de ano e mês

In [ ]:
df_devolucoes = transform.extract_date_parts(df_devolucoes, "data_devolucao")

df_devolucoes.select("data_devolucao", "ano", "mes").distinct().limit(5).toPandas()

# Arredonda a coluna de valor para 2 casas decimais

In [ ]:
value_cols = ["valor_devolvido"]
df_devolucoes = transform.round_values(df_devolucoes, value_cols, decimals=2)

df_devolucoes.select(*value_cols).limit(5).toPandas()

# Adiciona a data de carga da transformação

In [ ]:
# Adiciona a data de carga do processamento da Silver
df_devolucoes = df_devolucoes.withColumn("data_carga", F.to_date(F.lit(utils.get_current_date_str())))

df_devolucoes.select("devolucao_id", "data_carga").limit(5).toPandas()

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)